### Import libraries


In [1]:
import rasterio
import pandas as pd
import numpy as np

### Load the cleaned lake dataset

In [2]:
df = pd.read_csv("../data/processed/pakistan_full_training_dataset_clean.csv")
df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (8806, 7)


### Open the SRTM elevation raster and extract values

In [3]:
with rasterio.open("../data/raw/output_SRTMGL1.tif") as src:

    # Read the elevation band ONCE 
    elevation_data = src.read(1)

    elevations = []

    for _, row in df.iterrows():
        lon = row["longitude"]
        lat = row["latitude"]

        try:
            row_idx, col_idx = src.index(lon, lat)

            # Check whether coordinate falls inside the raster
            if (
                0 <= row_idx < elevation_data.shape[0]
                and 0 <= col_idx < elevation_data.shape[1]
            ):
                elevation = elevation_data[row_idx, col_idx]

                # Handle raster NoData values
                if src.nodata is not None and elevation == src.nodata:
                    elevation = np.nan

                elevations.append(elevation)
            else:
                elevations.append(np.nan)

        except Exception:
            elevations.append(np.nan)

### Add elevation to the dataset

In [4]:
df["elevation_m"] = elevations

### Check results

In [5]:
print(df[["latitude", "longitude", "elevation_m"]].head())

print("\nMissing elevation values:", df["elevation_m"].isna().sum())

print("\nElevation summary:")
print(df["elevation_m"].describe())

    latitude  longitude  elevation_m
0  34.828994  74.061891         3681
1  34.820079  74.086340         3663
2  34.806459  74.071990         3656
3  34.857497  74.076763         3680
4  34.861163  74.078262         3689

Missing elevation values: 0

Elevation summary:
count    8806.000000
mean     4187.092437
std       434.591369
min      2023.000000
25%      3939.000000
50%      4195.000000
75%      4469.750000
max      5541.000000
Name: elevation_m, dtype: float64


### Save the result

In [6]:
df.to_csv("../data/processed/glofguard_lakes_v1_elevation.csv", index=False)
print("\nSaved: glofguard_lakes_v1_elevation.csv")


Saved: glofguard_lakes_v1_elevation.csv
